In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_heatmap_prep AS
SELECT 
  -- 1. Safely convert your string Date into a real Timestamp format
  -- (Handles typical Chicago open-data configurations like 'MM/dd/yyyy hh:mm:ss a')
  CAST(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a') AS TIMESTAMP) AS clean_timestamp,

  -- 2. Extract hour integer (0-23) for your Y-Axis
  HOUR(CAST(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a') AS TIMESTAMP)) AS crime_hour,
  
  -- 3. Extract the 3-letter day name (Mon, Tue, Wed...) for your X-Axis
  date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') AS day_of_week,
  
  -- 4. Extract numerical index for sorting (Monday=1, Sunday=7) so order is perfect
  date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'u') AS day_sort_idx
FROM 
  workspace.default.cleaned_chicago_crimes -- <--- Change this to your actual Databricks delta table name
WHERE 
  `Date` IS NOT NULL;

In [0]:
%sql
SELECT 
  date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') AS day_of_week,
  HOUR(CAST(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a') AS TIMESTAMP)) AS crime_hour,
  COUNT(*) AS total_crimes
FROM 
  workspace.default.cleaned_chicago_crimes
WHERE 
  `Date` IS NOT NULL 
  AND to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a') IS NOT NULL
GROUP BY 
  1, 2
ORDER BY 
  CASE 
    WHEN date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') = 'Mon' THEN 1
    WHEN date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') = 'Tue' THEN 2
    WHEN date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') = 'Wed' THEN 3
    WHEN date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') = 'Thu' THEN 4
    WHEN date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') = 'Fri' THEN 5
    WHEN date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') = 'Sat' THEN 6
    WHEN date_format(to_timestamp(`Date`, 'MM/dd/yyyy hh:mm:ss a'), 'E') = 'Sun' THEN 7
  END ASC,
  crime_hour ASC;

day_of_week,crime_hour,total_crimes
Mon,0,64005
Mon,1,31179
Mon,2,25168
Mon,3,19857
Mon,4,15682
Mon,5,14574
Mon,6,18554
Mon,7,28188
Mon,8,41876
Mon,9,51549


In [0]:
%sql
ALTER TABLE workspace.default.cleaned_chicago_crimes 
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

In [0]:
%python
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

# 1. Load the clean dataset and drop rows with missing coordinates
df = spark.table("workspace.default.cleaned_chicago_crimes").filter(
    col("Latitude").isNotNull() & col("Longitude").isNotNull()
)

# 2. Convert Latitude and Longitude into a ML feature vector coordinate matrix
assembler = VectorAssembler(inputCols=["Longitude", "Latitude"], outputCol="features")
ml_data = assembler.transform(df)

# 3. Train the K-Means Model (Setting K=5 clusters)
kmeans = KMeans().setK(5).setSeed(42).setFeaturesCol("features").setPredictionCol("Cluster")
model = kmeans.fit(ml_data)

# 4. Apply the model to assign a Cluster ID (0, 1, 2, 3, 4) to every crime row
clustered_df = model.transform(ml_data).drop("features")

# 5. Save this as a new physical table that includes the 'Cluster' column
# 5. Save this as a new physical table while forcing Delta Lake to allow spaces
(clustered_df.write
 .mode("overwrite")
 .option("delta.columnMapping.mode", "name")  # <-- This fixes the space error!
 .saveAsTable("workspace.default.chicago_crimes_clustered"))

print("Success! The clustered table has been created with space-safe column mapping.")

# 1. Load the clean dataset and drop rows with missing coordinates
df = spark.table("workspace.default.cleaned_chicago_crimes").filter(
    col("Latitude").isNotNull() & col("Longitude").isNotNull()
)

# 2. Convert Latitude and Longitude into a ML feature vector coordinate matrix
assembler = VectorAssembler(inputCols=["Longitude", "Latitude"], outputCol="features")
ml_data = assembler.transform(df)

# 3. Train the K-Means Model (Setting K=5 clusters)
kmeans = KMeans().setK(5).setSeed(42).setFeaturesCol("features").setPredictionCol("Cluster")
model = kmeans.fit(ml_data)

# 4. Apply the model to assign a Cluster ID (0, 1, 2, 3, 4) to every crime row
clustered_df = model.transform(ml_data).drop("features")

# 5. Save this as a new physical table that includes the 'Cluster' column
clustered_df.write.mode("overwrite").saveAsTable("workspace.default.chicago_crimes_clustered")
print("Success! The clustered table has been created.")

Success! The clustered table has been created with space-safe column mapping.
Success! The clustered table has been created.


In [0]:
%python
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

# 1. Load data and assemble spatial features
df = spark.table("workspace.default.cleaned_chicago_crimes").filter(
    col("Latitude").isNotNull() & col("Longitude").isNotNull()
)
assembler = VectorAssembler(inputCols=["Longitude", "Latitude"], outputCol="features")
ml_data = assembler.transform(df)

# 2. Loop through K values 2 to 8 (Starting at 2 avoids the K=1 error!)
elbow_data = []
for k in range(2, 9):
    kmeans = KMeans().setK(k).setSeed(42).setFeaturesCol("features")
    model = kmeans.fit(ml_data)
    
    # Get the summary metrics to extract inertia (WCSSE)
    summary = model.summary
    inertia = summary.trainingCost
    
    elbow_data.append((k, float(inertia)))

# 3. Save the results into your evaluation table
schema = ["k_value", "inertia_value"]
summary_df = spark.createDataFrame(elbow_data, schema)
summary_df.write.mode("overwrite").saveAsTable("workspace.default.chicago_kmeans_evaluation_summary")

print("Success! The evaluation table has been created.")

Success! The evaluation table has been created.
